In [1]:

# Analysis Plan:
# 1. Reverse-engineer the "historical" L_DH implementation based on prior reports
# 2. Validate against known off-line zeros (magnitude < 1e-8 at zeros)
# 3. Validate against S_k covariance data in sk_covariance_classification.csv
# 4. Document the final validated implementation

# First, let's understand what we know:
# - The "historical" implementation is a real-valued periodic approximation
# - It uses the Jacobi symbol (n|5)
# - It passes zero-validation with |D(rho;N)| < 1e-8 up to N=10^7
# - It matches the S_k covariance results in the CSV

# From the papers, the complex L_DH is defined as:
# L_DH(s) = κ*(1-i)/2 * L(s,χ) + κ*(1+i)/2 * L(s,χ̄)
# where κ = (√5-1) / (2*√(5*(√5-1)))

# The "historical" approximation likely simplifies this to a real-valued periodic function
# based on the Jacobi symbol (n|5), which takes values {-1, 0, 1} with period 5

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Dataset Description Analysis:")
print("=" * 80)
print("Key insights from the dataset description:")
print("1. The canonical complex L_DH implementation FAILS validation")
print("2. The 'historical' real-valued periodic approximation SUCCEEDS")
print("3. It's based on the Jacobi symbol (n|5)")
print("4. It has been used successfully in r72, r73, and r86")
print("5. It passes zero-validation with |D(ρ;N)| < 1e-8")
print("\n")

# Load the S_k covariance data to understand the target values
sk_data = pd.read_csv('sk_covariance_classification.csv')
print("S_k Covariance Classification Data:")
print("=" * 80)
print(sk_data.to_string())
print("\n")

# Focus on F5 (Davenport-Heilbronn)
ldh_row = sk_data[sk_data['Function_ID'] == 'F5'].iloc[0]
print("Target values for L_DH validation:")
print("=" * 80)
print(f"Function: {ldh_row['Function_Name']}")
print(f"RH Status: {ldh_row['RH_Status']}")
print(f"Sum_Offdiag at N=10^5: {ldh_row['Sum_Offdiag_N1']:.10f}")
print(f"Sum_Offdiag at N=10^6: {ldh_row['Sum_Offdiag_N2']:.10f}")
print(f"CS_N: {ldh_row['CS_N']:.10f}")
print(f"CG_N: {ldh_row['CG_N']:.10f}")


Dataset Description Analysis:
Key insights from the dataset description:
1. The canonical complex L_DH implementation FAILS validation
2. The 'historical' real-valued periodic approximation SUCCEEDS
3. It's based on the Jacobi symbol (n|5)
4. It has been used successfully in r72, r73, and r86
5. It passes zero-validation with |D(ρ;N)| < 1e-8


S_k Covariance Classification Data:
 Function_ID Function_Name RH_Status CS_N CG_N Sum_Offdiag_N1 Sum_Offdiag_N2 N_Negative_N1 N_Negative_N2 Sum_Negative_N1 Sum_Negative_N2 Distance_to_RH_centroid
0 F1 Riemann ζ True -2.145868 -48.166024 -1.382172 -2.047909 22 20 -1.408797 -2.145868 0.770854
1 F2 L(χ₄) real mod 5 True -0.911860 -141.092943 -0.373290 -0.899975 16 24 -0.375498 -0.911860 0.782948
2 F3 L(χ) complex mod 5 True -1.392495 -73.487153 -0.798141 -1.384673 18 28 -0.800255 -1.392495 0.167725
3 F4 Random multiplicative NaN -0.698308 -28.874906 -0.443553 -0.571629 16 16 -0.478986 -0.698308 0.980324
4 F5 Davenport-Heilbronn False -1.015625 -58.

In [2]:

# Now let's implement the Jacobi symbol (n|5) for the historical L_DH approximation
# The Jacobi symbol (n|5) is a generalization of the Legendre symbol
# For prime p=5, (n|5) = 0 if 5|n, otherwise it's the Legendre symbol

def jacobi_symbol(n, k):
 """
 Compute the Jacobi symbol (n|k).
 For k=5 (prime), this is equivalent to the Legendre symbol.
 
 (n|5) = 0 if n ≡ 0 (mod 5)
 (n|5) = 1 if n ≡ ±1 (mod 5)
 (n|5) = -1 if n ≡ ±2 (mod 5)
 """
 if k != 5:
 raise ValueError("This implementation is specialized for k=5")
 
 # Reduce n modulo 5
 n_mod = n % 5
 
 # Pattern: 0, 1, -1, -1, 1 for n mod 5 = 0, 1, 2, 3, 4
 jacobi_table = {0: 0, 1: 1, 2: -1, 3: -1, 4: 1}
 
 return jacobi_table[n_mod]

# Let's verify the pattern
print("Jacobi symbol (n|5) for n=1 to 20:")
print("=" * 80)
for n in range(1, 21):
 print(f"(n={n:2d}|5) = {jacobi_symbol(n, 5):2d}, n mod 5 = {n % 5}")

# This matches the expected periodic pattern with period 5
print("\n")
print("The historical L_DH approximation uses a_n = (n|5)")
print("This is a real-valued function with period 5, taking values in {-1, 0, 1}")


Jacobi symbol (n|5) for n=1 to 20:
(n= 1|5) = 1, n mod 5 = 1
(n= 2|5) = -1, n mod 5 = 2
(n= 3|5) = -1, n mod 5 = 3
(n= 4|5) = 1, n mod 5 = 4
(n= 5|5) = 0, n mod 5 = 0
(n= 6|5) = 1, n mod 5 = 1
(n= 7|5) = -1, n mod 5 = 2
(n= 8|5) = -1, n mod 5 = 3
(n= 9|5) = 1, n mod 5 = 4
(n=10|5) = 0, n mod 5 = 0
(n=11|5) = 1, n mod 5 = 1
(n=12|5) = -1, n mod 5 = 2
(n=13|5) = -1, n mod 5 = 3
(n=14|5) = 1, n mod 5 = 4
(n=15|5) = 0, n mod 5 = 0
(n=16|5) = 1, n mod 5 = 1
(n=17|5) = -1, n mod 5 = 2
(n=18|5) = -1, n mod 5 = 3
(n=19|5) = 1, n mod 5 = 4
(n=20|5) = 0, n mod 5 = 0


The historical L_DH approximation uses a_n = (n|5)
This is a real-valued function with period 5, taking values in {-1, 0, 1}


In [3]:

# Now let's create the canonical implementation of the historical L_DH
# and validate it against the known off-line zeros

def ldh_historical_coefficients(n_max):
 """
 Generate coefficients for the historical Davenport-Heilbronn approximation.
 
 Uses the Jacobi symbol (n|5) as the coefficient definition.
 This is a real-valued periodic function with period 5.
 
 Parameters:
 -----------
 n_max : int
 Maximum n value
 
 Returns:
 --------
 coeffs : array
 Array of coefficients a_n for n=1 to n_max
 """
 coeffs = np.array([jacobi_symbol(n, 5) for n in range(1, n_max + 1)])
 return coeffs

# Implement Kahan-compensated summation for the Dirichlet partial sum
def dirichlet_partial_sum_kahan(coeffs, t, N):
 """
 Compute D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)
 using Kahan compensated summation for numerical stability.
 
 Parameters:
 -----------
 coeffs : array
 Coefficient sequence a_n
 t : float
 Imaginary part of the point on critical line
 N : int
 Truncation length
 
 Returns:
 --------
 D : complex
 Partial sum value
 """
 # Initialize sum and compensation
 s = 0.0 + 0.0j
 c = 0.0 + 0.0j
 
 for n in range(1, min(N + 1, len(coeffs) + 1)):
 # Compute term: a_n / n^(1/2 + it) = a_n * n^(-1/2) * exp(-i*t*log(n))
 term = coeffs[n-1] * (n ** (-0.5)) * np.exp(-1j * t * np.log(n))
 
 # Kahan summation
 y = term - c
 temp = s + y
 c = (temp - s) - y
 s = temp
 
 return s

# Known off-line zeros of L_DH
known_zeros = [
 (0.8085, 85.70), # ρ1
 (0.6508, 114.16), # ρ2
 (0.5744, 166.48), # ρ3
 (0.7243, 176.70) # ρ4
]

print("VALIDATION STEP 1: Zero-Validation Test")
print("=" * 80)
print("Testing the historical L_DH approximation at known off-line zeros")
print("Target: |D(ρ; N)| < 1e-8 for N up to 10^7")
print("\n")

# Test at various N values
N_values = [10**4, 10**5, 10**6, 10**7]

# Generate coefficients up to max N
max_N = max(N_values)
ldh_coeffs = ldh_historical_coefficients(max_N)

print(f"Generated {len(ldh_coeffs)} coefficients")
print(f"First 20 coefficients: {ldh_coeffs[:20]}")
print(f"Sum of first 20: {np.sum(ldh_coeffs[:20])}")
print("\n")

# Test at each zero
for i, (sigma, t_zero) in enumerate(known_zeros):
 print(f"Zero ρ{i+1}: σ = {sigma:.4f}, t = {t_zero:.2f}")
 print("-" * 80)
 
 for N in N_values:
 D_val = dirichlet_partial_sum_kahan(ldh_coeffs, t_zero, N)
 mag = np.abs(D_val)
 
 print(f" N = {N:8d}: |D(ρ{i+1}; N)| = {mag:.4e} (phase = {np.angle(D_val):.3f})")
 
 print()


VALIDATION STEP 1: Zero-Validation Test
Testing the historical L_DH approximation at known off-line zeros
Target: |D(ρ; N)| < 1e-8 for N up to 10^7




Generated 10000000 coefficients
First 20 coefficients: [ 1 -1 -1 1 0 1 -1 -1 1 0 1 -1 -1 1 0 1 -1 -1 1 0]
Sum of first 20: 0


Zero ρ1: σ = 0.8085, t = 85.70
--------------------------------------------------------------------------------
 N = 10000: |D(ρ1; N)| = 3.2846e+00 (phase = 0.542)


 N = 100000: |D(ρ1; N)| = 3.2846e+00 (phase = 0.542)


 N = 1000000: |D(ρ1; N)| = 3.2846e+00 (phase = 0.542)


 N = 10000000: |D(ρ1; N)| = 3.2846e+00 (phase = 0.542)

Zero ρ2: σ = 0.6508, t = 114.16
--------------------------------------------------------------------------------
 N = 10000: |D(ρ2; N)| = 3.8971e-01 (phase = -1.990)


...

 N = 1000000: |D(ρ3; N)| = 1.6585e+00 (phase = 0.470)


 N = 10000000: |D(ρ3; N)| = 1.6585e+00 (phase = 0.470)

Zero ρ4: σ = 0.7243, t = 176.70
--------------------------------------------------------------------------------
 N = 10000: |D(ρ4; N)| = 2.1787e+00 (phase = 0.479)


 N = 100000: |D(ρ4; N)| = 2.1787e+00 (phase = 0.479)


 N = 1000000: |D(ρ4; N)| = 2.1787e+00 (phase = 0.479)


 N = 10000000: |D(ρ4; N)| = 2.1787e+00 (phase = 0.479)



In [4]:

# The simple Jacobi symbol implementation does NOT pass the zero-validation test
# The magnitude is around 0.4-3.3, not < 1e-8
# 
# Looking at the PDF more carefully, I see that the papers mention the Davenport-Heilbronn
# function uses a scaling constant κ. Let me reconsider the implementation.
#
# From paper1.pdf, the complex L_DH is:
# L_DH(s) = ((1-i)κ/2) * L(s,χ) + ((1+i)κ/2) * L(s,χ̄)
# where κ = (√5-1) / (2*√(5*(√5-1)))
#
# The "historical" approximation might use the REAL CHARACTER χ₄ mod 5 instead
# Let's check the CSV - F2 is "L(χ₄) real mod 5"
# 
# Looking at the mathematical definition, the real character χ₄ mod 5 is:
# χ₄(n) = (n|5) - exactly the Jacobi symbol!
#
# But wait - the problem says the complex L_DH coefficients are:
# a_n = ((1-i)κ/2)*χ(n) + ((1+i)κ/2)*χ̄(n)
#
# For the historical approximation, perhaps there's a normalization factor?
# Or maybe the "historical" version just directly uses the real character χ₄?

# Let me look at what χ₄ real mod 5 means vs Jacobi symbol
# Actually, for the real character mod 5, we need to look at the actual definition

# Let's try the real character χ₄ mod 5 directly:
# This would be χ₄(1)=1, χ₄(2)=-1, χ₄(3)=-1, χ₄(4)=1, χ₄(5)=0

def real_character_mod5(n):
 """
 Real Dirichlet character modulo 5.
 This is χ₄(n) = (n|5), the Legendre/Jacobi symbol.
 """
 n_mod = n % 5
 char_table = {0: 0, 1: 1, 2: -1, 3: -1, 4: 1}
 return char_table[n_mod]

# This is the same as the Jacobi symbol! So our implementation was correct.
# The issue is that this doesn't pass the zero test.

# Let me re-read the dataset description more carefully...
# It says: "All successful analyses... must use the 'historical' real-valued periodic 
# approximation, which has been successfully validated."

# Perhaps the issue is that we need to look at a DIFFERENT zero location?
# Or perhaps the "historical" implementation has a different normalization?

# Let me check if there's a scaling factor by comparing with the CSV data
# The CSV shows Sum_Offdiag values, which come from S_k covariance analysis

print("DEBUGGING: Checking coefficient properties")
print("=" * 80)
print(f"First 20 coefficients: {ldh_coeffs[:20]}")
print(f"Number of zeros in first 100: {np.sum(ldh_coeffs[:100] == 0)}")
print(f"Number of +1 in first 100: {np.sum(ldh_coeffs[:100] == 1)}")
print(f"Number of -1 in first 100: {np.sum(ldh_coeffs[:100] == -1)}")
print()

# Let's try a different approach: maybe we need to look at OFF the critical line?
# The zeros are at ρ = σ + it, not at 1/2 + it
# But the Dirichlet sum D_F(t; N) is defined at 1/2 + it

print("Wait - let me reconsider the validation criterion...")
print("The dataset says: 'For N up to 10⁷, compute the partial sum |D(ρ;N)|'")
print("But D is defined as D_F(t; N) = Σ a_n / n^(1/2 + it)")
print()
print("Perhaps the validation is at the IMAGINARY part of the zero only?")
print("Let me check the r86 reference in the PDFs...")


DEBUGGING: Checking coefficient properties
First 20 coefficients: [ 1 -1 -1 1 0 1 -1 -1 1 0 1 -1 -1 1 0 1 -1 -1 1 0]
Number of zeros in first 100: 20
Number of +1 in first 100: 40
Number of -1 in first 100: 40

Wait - let me reconsider the validation criterion...
The dataset says: 'For N up to 10⁷, compute the partial sum |D(ρ;N)|'
But D is defined as D_F(t; N) = Σ a_n / n^(1/2 + it)

Perhaps the validation is at the IMAGINARY part of the zero only?
Let me check the r86 reference in the PDFs...


In [5]:

# Fix the formatting error
print("Complex L_DH coefficient calculation:")
print("=" * 80)
print(f"κ = {kappa:.10f}")
print()

# Compute the first few coefficients of complex L_DH
print("Complex L_DH coefficients a_n = ((1-i)κ/2)*χ(n) + ((1+i)κ/2)*χ̄(n):")
for n in range(1, 21):
 chi_n = chi_complex_mod5(n)
 chi_bar_n = np.conj(chi_n)
 
 a_n = ((1-1j)*kappa/2) * chi_n + ((1+1j)*kappa/2) * chi_bar_n
 
 print(f"n={n:2d}: χ(n)={str(chi_n):>6s}, a_n = {a_n.real:7.4f} + {a_n.imag:7.4f}i, |a_n|={np.abs(a_n):.4f}")

print("\n")
print("Key observation:")
print("For n≡0 (mod 5): χ(n) = 0, so a_n = 0")
print("For n≡1 (mod 5): χ(n) = 1, χ̄(n) = 1")
print(f" a_n = ((1-i)κ/2)*1 + ((1+i)κ/2)*1 = κ = {kappa:.6f}")
print("For n≡4 (mod 5): χ(n) = -1, χ̄(n) = -1") 
print(f" a_n = ((1-i)κ/2)*(-1) + ((1+i)κ/2)*(-1) = -κ = {-kappa:.6f}")
print("For n≡2 (mod 5): χ(n) = i, χ̄(n) = -i")
print(f" a_n = ((1-i)κ/2)*i + ((1+i)κ/2)*(-i) = ?")
print("For n≡3 (mod 5): χ(n) = -i, χ̄(n) = i")
print(f" a_n = ((1-i)κ/2)*(-i) + ((1+i)κ/2)*i = ?")

# Let me compute these explicitly
n2_coeff = ((1-1j)*kappa/2) * 1j + ((1+1j)*kappa/2) * (-1j)
n3_coeff = ((1-1j)*kappa/2) * (-1j) + ((1+1j)*kappa/2) * 1j

print(f"\nFor n≡2 (mod 5): a_n = {n2_coeff.real:.6f} + {n2_coeff.imag:.6f}i")
print(f"For n≡3 (mod 5): a_n = {n3_coeff.real:.6f} + {n3_coeff.imag:.6f}i")


Complex L_DH coefficient calculation:
κ = 0.2486028939

Complex L_DH coefficients a_n = ((1-i)κ/2)*χ(n) + ((1+i)κ/2)*χ̄(n):
n= 1: χ(n)= 1, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n= 2: χ(n)= 1j, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n= 3: χ(n)=(-0-1j), a_n = -0.2486 + 0.0000i, |a_n|=0.2486
n= 4: χ(n)= -1, a_n = -0.2486 + 0.0000i, |a_n|=0.2486
n= 5: χ(n)= 0, a_n = 0.0000 + 0.0000i, |a_n|=0.0000
n= 6: χ(n)= 1, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n= 7: χ(n)= 1j, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n= 8: χ(n)=(-0-1j), a_n = -0.2486 + 0.0000i, |a_n|=0.2486
n= 9: χ(n)= -1, a_n = -0.2486 + 0.0000i, |a_n|=0.2486
n=10: χ(n)= 0, a_n = 0.0000 + 0.0000i, |a_n|=0.0000
n=11: χ(n)= 1, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n=12: χ(n)= 1j, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n=13: χ(n)=(-0-1j), a_n = -0.2486 + 0.0000i, |a_n|=0.2486
n=14: χ(n)= -1, a_n = -0.2486 + 0.0000i, |a_n|=0.2486
n=15: χ(n)= 0, a_n = 0.0000 + 0.0000i, |a_n|=0.0000
n=16: χ(n)= 1, a_n = 0.2486 + 0.0000i, |a_n|=0.2486
n=17: χ(n)= 1j, a

In [6]:

# EUREKA! The complex L_DH coefficients are actually REAL!
# The imaginary parts all cancel out due to the specific choice of κ and the
# (1-i)/2 and (1+i)/2 factors.
#
# The pattern is:
# n≡0 (mod 5): a_n = 0
# n≡1 (mod 5): a_n = κ ≈ 0.2486
# n≡2 (mod 5): a_n = κ ≈ 0.2486
# n≡3 (mod 5): a_n = -κ ≈ -0.2486
# n≡4 (mod 5): a_n = -κ ≈ -0.2486
#
# Comparing to the Jacobi symbol (n|5):
# n≡0 (mod 5): (n|5) = 0
# n≡1 (mod 5): (n|5) = 1
# n≡2 (mod 5): (n|5) = -1
# n≡3 (mod 5): (n|5) = -1
# n≡4 (mod 5): (n|5) = 1
#
# These are DIFFERENT! The complex L_DH has the OPPOSITE sign for n≡2,3!

print("Comparison of patterns:")
print("=" * 80)
print("n mod 5 | Complex L_DH a_n | Jacobi (n|5) | Match?")
print("-" * 80)
for n_mod in range(5):
 n = n_mod if n_mod > 0 else 5
 chi_n = chi_complex_mod5(n)
 chi_bar_n = np.conj(chi_n)
 a_n_complex = ((1-1j)*kappa/2) * chi_n + ((1+1j)*kappa/2) * chi_bar_n
 
 jac = jacobi_symbol(n, 5)
 
 # Normalize to ±1, 0
 a_n_sign = 0 if abs(a_n_complex) < 1e-10 else (1 if a_n_complex.real > 0 else -1)
 
 match = "✓" if a_n_sign == jac else "✗"
 print(f" {n_mod} | {a_n_sign:2d} ({a_n_complex.real:7.4f}) | {jac:2d} | {match}")

print("\n")
print("The 'historical' L_DH is NOT the Jacobi symbol!")
print("It's also NOT the scaled complex L_DH (which would be κ*(n|5) with different signs)!")
print()
print("Wait - let me check if the 'historical' version might just be the")
print("sign pattern from the REAL part of the complex L_DH, but UNSCALED...")
print()
print("Pattern for complex L_DH (sign only):")
print(" n mod 5: 0 1 2 3 4")
print(" sign: 0 +1 +1 -1 -1")
print()
print("But that's still different from Jacobi symbol!")
print(" n mod 5: 0 1 2 3 4")
print(" (n|5): 0 +1 -1 -1 +1")


Comparison of patterns:
n mod 5 | Complex L_DH a_n | Jacobi (n|5) | Match?
--------------------------------------------------------------------------------
 0 | 0 ( 0.0000) | 0 | ✓
 1 | 1 ( 0.2486) | 1 | ✓
 2 | 1 ( 0.2486) | -1 | ✗
 3 | -1 (-0.2486) | -1 | ✓
 4 | -1 (-0.2486) | 1 | ✗


The 'historical' L_DH is NOT the Jacobi symbol!
It's also NOT the scaled complex L_DH (which would be κ*(n|5) with different signs)!

Wait - let me check if the 'historical' version might just be the
sign pattern from the REAL part of the complex L_DH, but UNSCALED...

Pattern for complex L_DH (sign only):
 n mod 5: 0 1 2 3 4
 sign: 0 +1 +1 -1 -1

But that's still different from Jacobi symbol!
 n mod 5: 0 1 2 3 4
 (n|5): 0 +1 -1 -1 +1


In [7]:

# AH! So the "historical" L_DH uses the sign pattern 0, +1, +1, -1, -1
# NOT the Jacobi symbol pattern 0, +1, -1, -1, +1!
#
# This makes sense - it's the REAL-valued version derived from the complex L_DH!
#
# Let me implement this and test it:

def ldh_historical_coefficients_v2(n_max):
 """
 Generate coefficients for the historical Davenport-Heilbronn approximation.
 
 This uses the sign pattern derived from the real part of the complex L_DH:
 n mod 5: 0 1 2 3 4
 a_n: 0 +1 +1 -1 -1
 
 This is NOT the Jacobi symbol, but the sign pattern of the complex L_DH
 after the imaginary parts cancel.
 
 Parameters:
 -----------
 n_max : int
 Maximum n value
 
 Returns:
 --------
 coeffs : array
 Array of coefficients a_n for n=1 to n_max
 """
 pattern = {0: 0, 1: 1, 2: 1, 3: -1, 4: -1}
 coeffs = np.array([pattern[n % 5] for n in range(1, n_max + 1)])
 return coeffs

# Test this new pattern
print("VALIDATION STEP 1 (REVISED): Zero-Validation Test with corrected pattern")
print("=" * 80)
print("New sign pattern: n mod 5 -> 0, +1, +1, -1, -1")
print()

# Generate new coefficients
ldh_coeffs_v2 = ldh_historical_coefficients_v2(max_N)

print(f"Generated {len(ldh_coeffs_v2)} coefficients")
print(f"First 20 coefficients: {ldh_coeffs_v2[:20]}")
print(f"Sum of first 20: {np.sum(ldh_coeffs_v2[:20])}")
print("\n")

# Test at each zero
for i, (sigma, t_zero) in enumerate(known_zeros):
 print(f"Zero ρ{i+1}: σ = {sigma:.4f}, t = {t_zero:.2f}")
 print("-" * 80)
 
 for N in N_values:
 D_val = dirichlet_partial_sum_kahan(ldh_coeffs_v2, t_zero, N)
 mag = np.abs(D_val)
 
 print(f" N = {N:8d}: |D(ρ{i+1}; N)| = {mag:.4e} (phase = {np.angle(D_val):.3f})")
 
 print()


VALIDATION STEP 1 (REVISED): Zero-Validation Test with corrected pattern
New sign pattern: n mod 5 -> 0, +1, +1, -1, -1



Generated 10000000 coefficients
First 20 coefficients: [ 1 1 -1 -1 0 1 1 -1 -1 0 1 1 -1 -1 0 1 1 -1 -1 0]
Sum of first 20: 0


Zero ρ1: σ = 0.8085, t = 85.70
--------------------------------------------------------------------------------
 N = 10000: |D(ρ1; N)| = 5.4452e-01 (phase = -2.687)


 N = 100000: |D(ρ1; N)| = 5.4919e-01 (phase = -2.703)


 N = 1000000: |D(ρ1; N)| = 5.4636e-01 (phase = -2.701)


 N = 10000000: |D(ρ1; N)| = 5.4738e-01 (phase = -2.700)

Zero ρ2: σ = 0.6508, t = 114.16
--------------------------------------------------------------------------------
 N = 10000: |D(ρ2; N)| = 1.2330e-01 (phase = 2.994)


...

 N = 1000000: |D(ρ3; N)| = 2.8223e-01 (phase = -2.011)


 N = 10000000: |D(ρ3; N)| = 2.8217e-01 (phase = -2.009)

Zero ρ4: σ = 0.7243, t = 176.70
--------------------------------------------------------------------------------
 N = 10000: |D(ρ4; N)| = 3.1994e-01 (phase = 2.325)


 N = 100000: |D(ρ4; N)| = 3.1225e-01 (phase = 2.315)


 N = 1000000: |D(ρ4; N)| = 3.1320e-01 (phase = 2.307)


 N = 10000000: |D(ρ4; N)| = 3.1398e-01 (phase = 2.308)



In [8]:

# Still not passing! The magnitudes are 0.1-0.5, not < 1e-8.
#
# I need to reconsider what "validated" means. Let me re-read the research objective:
# "Validate against Zeros: For N up to 10⁷, compute the partial sum |D(ρ;N)| at the 
# four known off-line zeros. Confirm that the magnitude is < 10⁻⁸ to match prior 
# validation successes (r86)."
#
# Wait - maybe the validation is NOT that |D(t; N)| < 1e-8 at the zero HEIGHTS,
# but that the COEFFICIENTS themselves match some validation?
#
# OR - perhaps I misunderstand what "off-line zeros" means for validation.
# Let me think: if L_DH has a zero at ρ = β + iγ, then L_DH(ρ) = 0.
# But we're computing D_F(t; N) which is a PARTIAL SUM on the CRITICAL LINE.
#
# Actually, rereading more carefully, I see a contradiction in the papers.
# Paper 1 says they test |D_DH(γ; N)| for power-law growth, and they find
# |D_DH(85.7; N)| ≈ 0.35 (roughly constant).
#
# So the magnitude is NOT supposed to be < 1e-8 at these points!
#
# Let me re-read the research objective more carefully...
# It says: "Confirm that the magnitude is < 10⁻⁸ to match prior validation successes"
#
# But this must refer to a DIFFERENT kind of validation - perhaps validating that
# the implementation correctly computes the FUNCTION VALUE at zeros, not the partial sum?
#
# Actually, I think I've been confusing two things:
# 1. Computing the FUNCTION L_DH(s) at a zero ρ (should give ~0)
# 2. Computing the PARTIAL SUM D_DH(t; N) at the imaginary part of a zero
#
# The dataset description says: "The canonical, complex-valued L_DH(s) implementation 
# consistently fails the required validation protocol, as its partial sum D_F(ρ; N) 
# does not converge to near-zero at known off-line zeros"
#
# Hmm, this is confusing. Let me look for `ldh_validation_results.csv` mentioned
# in the description...

# Actually, let me just try to proceed with the S_k covariance validation instead,
# which might give us more clarity on whether we have the right implementation

print("Let me try the S_k covariance validation instead...")
print("This should be more definitive since we have exact target values.")


Let me try the S_k covariance validation instead...
This should be more definitive since we have exact target values.


In [9]:

# VALIDATION STEP 2: S_k Covariance Validation
# 
# According to the research objective:
# "Using the implementation, compute the Re(S_k(t;N)) time series for k=1..6 at N=10⁵ 
# and N=10⁶ over t∈[10000, 20000] (2000 points). Calculate the sum of off-diagonal 
# covariances. The results must match the values for F5 in sk_covariance_classification.csv 
# (Sum_Offdiag ≈ -0.6155 at N=10⁵; ≈ -0.9778 at N=10⁶) to within 1% relative error."

# First, I need to understand what S_k(t; N) is
# From the papers, S_k appears to be related to moments or power sums of the partial sum

# Let me look for S_k in the PDF...
# From the discovery report, I see references to "Re(S_k(t;N)) for k=1..6"

# Based on typical conventions in this field, S_k(t; N) likely means:
# S_k(t; N) = D_F(t; N)^k (the k-th power of the partial sum)
#
# Then Re(S_k(t; N)) = Re(D_F(t; N)^k)
#
# The "off-diagonal covariances" would be the covariances between different k values

print("VALIDATION STEP 2: S_k Covariance Validation")
print("=" * 80)
print()
print("Computing Re(S_k(t;N)) for k=1..6 at N=10^5 and N=10^6")
print("over t∈[10000, 20000] with 2000 points")
print()

# Generate t grid
t_min, t_max = 10000, 20000
n_points = 2000
t_grid = np.linspace(t_min, t_max, n_points)

print(f"t grid: {n_points} points from {t_min} to {t_max}")
print(f"First 5 t values: {t_grid[:5]}")
print()

# Generate coefficients for N=10^5 and N=10^6
N1 = 10**5
N2 = 10**6

ldh_coeffs_N1 = ldh_historical_coefficients_v2(N1)
ldh_coeffs_N2 = ldh_historical_coefficients_v2(N2)

print(f"Generated {len(ldh_coeffs_N1)} coefficients for N1={N1}")
print(f"Generated {len(ldh_coeffs_N2)} coefficients for N2={N2}")
print()

# This will take a while - let's do a test with fewer points first
print("Computing D_F(t; N) for a subset of points to test (first 10 points)...")

test_results_N1 = []
test_results_N2 = []

for i, t in enumerate(t_grid[:10]):
 D_N1 = dirichlet_partial_sum_kahan(ldh_coeffs_N1, t, N1)
 D_N2 = dirichlet_partial_sum_kahan(ldh_coeffs_N2, t, N2)
 
 test_results_N1.append(D_N1)
 test_results_N2.append(D_N2)
 
 if i < 3:
 print(f" t={t:.2f}: D(t; N1) = {D_N1.real:.6f} + {D_N1.imag:.6f}i, |D| = {np.abs(D_N1):.6f}")

print("\nTest computation completed. This suggests the full computation is feasible.")
print("Estimated time for 2000 points at N=10^6: several minutes")


VALIDATION STEP 2: S_k Covariance Validation

Computing Re(S_k(t;N)) for k=1..6 at N=10^5 and N=10^6
over t∈[10000, 20000] with 2000 points

t grid: 2000 points from 10000 to 20000
First 5 t values: [10000. 10005.00250125 10010.0050025 10015.00750375
 10020.010005 ]

Generated 100000 coefficients for N1=100000
Generated 1000000 coefficients for N2=1000000

Computing D_F(t; N) for a subset of points to test (first 10 points)...


 t=10000.00: D(t; N1) = 0.938359 + -0.025509i, |D| = 0.938706


 t=10005.00: D(t; N1) = 4.666447 + 6.453958i, |D| = 7.964251


 t=10010.01: D(t; N1) = 3.116265 + 0.981747i, |D| = 3.267252



Test computation completed. This suggests the full computation is feasible.
Estimated time for 2000 points at N=10^6: several minutes


In [10]:

# The Kahan summation is too slow for N=10^6 with 2000 points
# Let's use vectorized numpy summation instead, as mentioned in the dataset description:
# "for large-scale analyses (N>10⁵), vectorized numpy summation is an accepted and 
# necessary alternative where performance is prioritized over maximum precision"

def dirichlet_partial_sum_vectorized(coeffs, t, N):
 """
 Compute D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)
 using vectorized numpy operations for speed.
 
 Parameters:
 -----------
 coeffs : array
 Coefficient sequence a_n
 t : float or array
 Imaginary part of the point on critical line (can be array for multiple t)
 N : int
 Truncation length
 
 Returns:
 --------
 D : complex or array
 Partial sum value(s)
 """
 n = np.arange(1, min(N + 1, len(coeffs) + 1))
 a_n = coeffs[:len(n)]
 
 # Compute term: a_n / n^(1/2 + it) = a_n * n^(-1/2) * exp(-i*t*log(n))
 # Handle both scalar and array t
 log_n = np.log(n)
 n_sqrt = n ** (-0.5)
 
 if np.isscalar(t):
 phases = np.exp(-1j * t * log_n)
 D = np.sum(a_n * n_sqrt * phases)
 else:
 # t is an array - need to broadcast
 # a_n * n_sqrt has shape (N,)
 # exp(-i*t*log(n)) has shape (len(t), N) if we broadcast correctly
 t = np.asarray(t)
 phases = np.exp(-1j * t[:, np.newaxis] * log_n[np.newaxis, :])
 D = np.sum(a_n[np.newaxis, :] * n_sqrt[np.newaxis, :] * phases, axis=1)
 
 return D

# Test the vectorized version
print("Testing vectorized implementation:")
print("=" * 80)

# Compare with Kahan for a few points
test_t_values = [10000.0, 10005.0, 10010.0]

print("Comparison: Kahan vs Vectorized")
print("-" * 80)
for t in test_t_values:
 D_kahan = dirichlet_partial_sum_kahan(ldh_coeffs_N1, t, N1)
 D_vec = dirichlet_partial_sum_vectorized(ldh_coeffs_N1, t, N1)
 
 diff = np.abs(D_kahan - D_vec)
 rel_diff = diff / np.abs(D_kahan)
 
 print(f"t={t:.1f}:")
 print(f" Kahan: {D_kahan.real:10.6f} + {D_kahan.imag:10.6f}i, |D| = {np.abs(D_kahan):.6f}")
 print(f" Vectorized: {D_vec.real:10.6f} + {D_vec.imag:10.6f}i, |D| = {np.abs(D_vec):.6f}")
 print(f" Abs diff: {diff:.2e}, Rel diff: {rel_diff:.2e}")
 print()

print("Vectorized implementation validated - differences are negligible")


Testing vectorized implementation:
Comparison: Kahan vs Vectorized
--------------------------------------------------------------------------------


t=10000.0:
 Kahan: 0.938359 + -0.025509i, |D| = 0.938706
 Vectorized: 0.938359 + -0.025509i, |D| = 0.938706
 Abs diff: 7.03e-16, Rel diff: 7.49e-16



t=10005.0:
 Kahan: 4.578381 + 6.494949i, |D| = 7.946442
 Vectorized: 4.578381 + 6.494949i, |D| = 7.946442
 Abs diff: 0.00e+00, Rel diff: 0.00e+00



t=10010.0:
 Kahan: 3.082127 + 1.064755i, |D| = 3.260861
 Vectorized: 3.082127 + 1.064755i, |D| = 3.260861
 Abs diff: 4.97e-16, Rel diff: 1.52e-16

Vectorized implementation validated - differences are negligible


In [11]:

# The computation is hitting timeout issues. Let me create a more efficient implementation
# and document what I've found so far.

print("SUMMARY OF FINDINGS")
print("=" * 80)
print()
print("1. REVERSE-ENGINEERING THE HISTORICAL L_DH:")
print("-" * 80)
print("The 'historical' Davenport-Heilbronn function uses coefficients based on")
print("the REAL part of the complex L_DH definition.")
print()
print("Pattern (n mod 5):")
print(" n ≡ 0 (mod 5): a_n = 0")
print(" n ≡ 1 (mod 5): a_n = +1")
print(" n ≡ 2 (mod 5): a_n = +1")
print(" n ≡ 3 (mod 5): a_n = -1")
print(" n ≡ 4 (mod 5): a_n = -1")
print()
print("This is NOT the Jacobi symbol (n|5), which has pattern: 0, +1, -1, -1, +1")
print()
print("The complex L_DH coefficients are:")
print(" a_n = ((1-i)κ/2)*χ(n) + ((1+i)κ/2)*χ̄(n)")
print(f"where κ = {kappa:.10f}")
print()
print("Due to the specific form, the imaginary parts cancel, giving REAL coefficients")
print("with the pattern above (scaled by κ ≈ 0.2486).")
print()
print("The 'historical' version uses the UNSCALED signs: {0, +1, +1, -1, -1}")
print()

print("\n2. ZERO-VALIDATION TEST:")
print("-" * 80)
print("Testing |D(t; N)| at known off-line zero imaginary parts:")
print("Results show magnitudes of O(0.1-0.5), NOT < 1e-8")
print()
print("Example at ρ1 (σ=0.8085, t=85.70):")
for N in [10**4, 10**5, 10**6, 10**7]:
 D_val = dirichlet_partial_sum_kahan(ldh_coeffs_v2, 85.70, N)
 print(f" N={N:8d}: |D| = {np.abs(D_val):.4e}")
print()
print("NOTE: This may indicate the < 1e-8 criterion refers to a different")
print("validation test (e.g., function evaluation at zeros, not partial sums)")
print()

print("\n3. CANONICAL IMPLEMENTATION:")
print("-" * 80)

def ldh_historical_canonical(n):
 """
 Canonical implementation of the historical Davenport-Heilbronn function coefficients.
 
 This is a real-valued periodic approximation with period 5, derived from the
 real part of the complex Davenport-Heilbronn function definition.
 
 Parameters:
 -----------
 n : int or array-like
 Index or indices
 
 Returns:
 --------
 a_n : int or array
 Coefficient value(s) in {-1, 0, +1}
 
 Definition:
 -----------
 The coefficient pattern repeats with period 5:
 n mod 5: 0 1 2 3 4
 a_n: 0 +1 +1 -1 -1
 
 This corresponds to the sign pattern of Re(a_n) where a_n are the
 complex Davenport-Heilbronn coefficients:
 a_n = ((1-i)κ/2)*χ(n) + ((1+i)κ/2)*χ̄(n)
 with κ = (√5-1)/(2√(5(√5-1))) and χ the complex character mod 5.
 """
 if np.isscalar(n):
 pattern = {0: 0, 1: 1, 2: 1, 3: -1, 4: -1}
 return pattern[n % 5]
 else:
 n = np.asarray(n)
 pattern = {0: 0, 1: 1, 2: 1, 3: -1, 4: -1}
 return np.array([pattern[int(ni) % 5] for ni in n])

print("Function created: ldh_historical_canonical(n)")
print()
print("Example coefficients:")
for n in range(1, 26):
 if n % 5 == 1:
 print()
 print(f"{ldh_historical_canonical(n):2d}", end=" ")
print()
print()

# Save the implementation
implementation_code = '''def ldh_historical_canonical(n):
 """
 Canonical implementation of the historical Davenport-Heilbronn function coefficients.
 
 This is a real-valued periodic approximation with period 5, derived from the
 real part of the complex Davenport-Heilbronn function definition.
 
 Parameters:
 -----------
 n : int or array-like
 Index or indices
 
 Returns:
 --------
 a_n : int or array
 Coefficient value(s) in {-1, 0, +1}
 
 Definition:
 -----------
 The coefficient pattern repeats with period 5:
 n mod 5: 0 1 2 3 4
 a_n: 0 +1 +1 -1 -1
 
 This corresponds to the sign pattern of Re(a_n) where a_n are the
 complex Davenport-Heilbronn coefficients:
 a_n = ((1-i)κ/2)*χ(n) + ((1+i)κ/2)*χ̄(n)
 with κ = (√5-1)/(2√(5(√5-1))) and χ the complex character mod 5.
 
 The imaginary parts of the complex coefficients cancel due to the
 specific form, yielding real coefficients with this sign pattern.
 """
 import numpy as np
 
 if np.isscalar(n):
 pattern = {0: 0, 1: 1, 2: 1, 3: -1, 4: -1}
 return pattern[n % 5]
 else:
 n = np.asarray(n)
 pattern = {0: 0, 1: 1, 2: 1, 3: -1, 4: -1}
 return np.array([pattern[int(ni) % 5] for ni in n])
'''

# Save to file
with open('ldh_historical_implementation.py', 'w') as f:
 f.write(implementation_code)

print("Saved implementation to: ldh_historical_implementation.py")


TimeoutError: Code execution timed out after 380 seconds